In [1]:
from dotenv import load_dotenv
load_dotenv()
import json
from rag_helper import RAGBase
from ingest import load_faq_data, build_index
from openai import OpenAI

In [2]:
openai_client = OpenAI()

In [3]:
documents = load_faq_data()
index = build_index(documents)

In [ ]:
messages = [
    {"role": "user", 
     "content": "I just discovered the course. Can I join it?"
     }
]

response = openai_client.responses.create(
    model="gpt-5.4-mini",
    input=messages,
)

response.output_text

In [4]:
def search(query):
    boost_dict = {"question": 3.0, "section": 0.5}
    filter_dict = {"course": "llm-zoomcamp"}

    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict,
        filter_dict=filter_dict
    )

In [5]:
search_tool = {
    "type": "function",
    "name": "search",
    "description": "Search the FAQ database for entries matching the given query.",
    "parameters": {
        "type": "object",
        "properties": {
            "query": {
                "type": "string",
                "description": "Search query text to look up in the course FAQ."
            }
        },
        "required": ["query"],
        "additionalProperties": False
    }
}

In [ ]:
response = openai_client.responses.create(
    model="gpt-5.4-mini",
    input=messages,
    tools=[search_tool]
)

response.output

In [ ]:
call = response.output[0]

In [ ]:
call

In [ ]:
call.arguments

In [ ]:
args = json.loads(call.arguments)

In [ ]:
results = search(**args)

In [ ]:
#turn results into json
result_json = json.dumps(results, indent=2)

In [ ]:
#send results to llm
function_call_output= {
    "type": "function_call_output",
    "call_id": call.call_id,
    "output": result_json,
}

In [ ]:
# messages only includes the 1st call
messages

In [ ]:
#add the decision (to call search) to messages
messages.append(call)

In [ ]:
# add the output to messages
messages.append(function_call_output)

In [ ]:
# Now messages includes the entire history (steps 1 to 3)
messages

In [ ]:
# We can now make the second call 
response = openai_client.responses.create(
    model="gpt-5.4-mini",
    input=messages,
    tools=[search_tool],
)

response.output_text

In [ ]:
usage = response.usage
usage.input_tokens, usage.output_tokens

In [ ]:
def calculate_gpt54mini_price(input_tokens, output_tokens):
    INPUT_PRICE_PER_MILLION = 0.15
    OUTPUT_PRICE_PER_MILLION = 0.60

    input_cost = (input_tokens / 1_000_000) * INPUT_PRICE_PER_MILLION
    output_cost = (output_tokens / 1_000_000) * OUTPUT_PRICE_PER_MILLION
    total_cost = input_cost + output_cost

    return {
        "input_cost": input_cost,
        "output_cost": output_cost,
        "total_cost": total_cost,
    }

result = calculate_gpt54mini_price(652, 33)
print("Total cost: $", round(result["total_cost"], 8))

In [6]:
def make_call(call):
    args = json.loads(call.arguments)

    if call.name == "search":
        result = search(**args)

    result_json = json.dumps(result, indent=2)

    return {
        "type": "function_call_output",
        "call_id": call.call_id,
        "output": result_json,
    }

In [7]:
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function.
Use as many keywords from the user question as possible when making first requests.

Make multiple searches.

Try to expand your search by using new keywords
based on the results you get from the search.

At the end, ask if there are other areas that the user wants to explore.
""".strip()

question = 'I just discovered the course. Can I still join?'

# The agent's memory
messages = [
    {"role": "developer", "content": instructions},
    {"role": "user", "content": question},
]

response = openai_client.responses.create(
    model="gpt-5.4-mini",
    input=messages,
    tools=[search_tool],
)

In [8]:
response.output

[ResponseFunctionToolCall(arguments='{"query":"join course late enrollment discovered course can I still join"}', call_id='call_pKd98XyshTORn0yi7wea9Sjt', name='search', type='function_call', id='fc_079857ef4e8e528f006a3a95ef7698819295bf3aadafcf532d', namespace=None, status='completed'),
 ResponseFunctionToolCall(arguments='{"query":"late join course enrollment access new student"}', call_id='call_vac50kCbJW3JrM0fLkoKbUqM', name='search', type='function_call', id='fc_079857ef4e8e528f006a3a95ef76b08192b040d330dd47228c', namespace=None, status='completed'),
 ResponseFunctionToolCall(arguments='{"query":"can I still join the course after it started FAQ"}', call_id='call_wFlurxadfllZLBcBYsRUeESG', name='search', type='function_call', id='fc_079857ef4e8e528f006a3a95ef76bc8192a8702407dc23360b', namespace=None, status='completed')]

In [9]:
response.output[0]

ResponseFunctionToolCall(arguments='{"query":"join course late enrollment discovered course can I still join"}', call_id='call_pKd98XyshTORn0yi7wea9Sjt', name='search', type='function_call', id='fc_079857ef4e8e528f006a3a95ef7698819295bf3aadafcf532d', namespace=None, status='completed')

In [10]:
messages.extend(response.output)
has_function_calls = False

for item in response.output:
    if item.type == "function_call":
        print("function_call:", item.name, item.arguments)
        call_output = make_call(item)
        messages.append(call_output)
        has_function_calls = True

    elif item.type == "message":
        print("ASSISTANT:")
        print(item.content[0].text)

function_call: search {"query":"join course late enrollment discovered course can I still join"}
function_call: search {"query":"late join course enrollment access new student"}
function_call: search {"query":"can I still join the course after it started FAQ"}


In [13]:
response = openai_client.responses.create(
    model="gpt-5.4-mini",
    input=messages,
    tools=[search_tool],
)

In [16]:
response.output[0].content

[ResponseOutputText(annotations=[], text='Yes — you can still join.\n\nYou can start learning anytime, and the course materials/videos are available. The main caveat is that if you want a certificate, you need to submit your project while submissions are still open. Also, certificates are only awarded for finishing with a live cohort, not in self-paced mode.\n\nIf you’d like, I can also point you to the best place to start and the usual workflow. Are there other areas you want to explore?', type='output_text', logprobs=[])]

In [17]:
messages.extend(response.output)
has_function_calls = False

for item in response.output:
    if item.type == "function_call":
        print("function_call:", item.name, item.arguments)
        call_output = make_call(item)
        messages.append(call_output)
        has_function_calls = True

    elif item.type == "message":
        print("ASSISTANT:")
        print(item.content[0].text)

ASSISTANT:
Yes — you can still join.

You can start learning anytime, and the course materials/videos are available. The main caveat is that if you want a certificate, you need to submit your project while submissions are still open. Also, certificates are only awarded for finishing with a live cohort, not in self-paced mode.

If you’d like, I can also point you to the best place to start and the usual workflow. Are there other areas you want to explore?


In [18]:
messages

[{'role': 'developer',
  'content': "You're a course teaching assistant.\nYou're given a question from a course student and your task is to answer it.\n\nIf you want to look up information, use the search function.\nUse as many keywords from the user question as possible when making first requests.\n\nMake multiple searches.\n\nTry to expand your search by using new keywords\nbased on the results you get from the search.\n\nAt the end, ask if there are other areas that the user wants to explore."},
 {'role': 'user',
  'content': 'I just discovered the course. Can I still join?'},
 ResponseFunctionToolCall(arguments='{"query":"join course late enrollment discovered course can I still join"}', call_id='call_pKd98XyshTORn0yi7wea9Sjt', name='search', type='function_call', id='fc_079857ef4e8e528f006a3a95ef7698819295bf3aadafcf532d', namespace=None, status='completed'),
 ResponseFunctionToolCall(arguments='{"query":"late join course enrollment access new student"}', call_id='call_vac50kCbJW3J